# Построение матрицы ковариации и дисперсии в Python

## Теоретическое обоснование

### Что такое матрица ковариации-дисперсии?

Формальное определение: матрица ковариации-дисперсии - это матрица, элемент в позиции $i,j$ которой представляет собой ковариацию между $i$-м и $j$-м элементами вектора случайных величин.

Неформально можно сказать, что матрица ковариации-дисперсии - это матрица ковариаций, а поскольку ковариация случайной величины с самой собой является её дисперсией, главная диагональ матрицы заполнена дисперсиями случайных величин (отсюда и название).

Математически это можно представить как:

$$\Sigma = \begin{bmatrix}
\sigma_1^2 & \sigma_{12} & \cdots & \sigma_{1n} \\
\sigma_{21} & \sigma_2^2 & \cdots & \sigma_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
\sigma_{n1} & \sigma_{n2} & \cdots & \sigma_n^2
\end{bmatrix}$$

где $\sigma_i^2$ - дисперсия $i$-й случайной величины, а $\sigma_{ij}$ - ковариация между $i$-й и $j$-й случайными величинами.

### Для чего это полезно?

При расчете VaR (Value at Risk) для отдельной акции необходимо собрать стандартное отклонение доходностей этой акции. Однако при расчете VaR для портфеля все становится значительно сложнее, поскольку нельзя просто складывать или вычитать дисперсии. Более интуитивно можно сказать, что матрица ковариации-дисперсии обобщает понятие дисперсии на многомерный случай.

### Взаимосвязь между дисперсией, стандартным отклонением и ковариацией

Стандартное отклонение - это квадратный корень из дисперсии:

$$\sigma = \sqrt{\sigma^2}$$

Ковариация может быть получена с учетом корреляции и стандартных отклонений:

$$\sigma_{ij} = \rho_{ij} \cdot \sigma_i \cdot \sigma_j$$

где $\rho_{ij}$ - коэффициент корреляции между $i$-й и $j$-й случайными величинами.

## Практическая реализация на Python

In [ ]:
import numpy as np
import math

# Стандартные отклонения для акций
stdv = {"ABC": 0.3, "XYZ": 0.2}

# Тикеры для корреляционной матрицы
tickersCorr = ["ABC", "XYZ"]

# Матрица корреляции (предполагаем корреляцию 0.5)
c = [[1, 0.5], [0.5, 1]]

def varCovarMatrix(stocksInPortfolio):
    """
    Функция для создания матрицы ковариации-дисперсии
    
    Параметры:
    stocksInPortfolio - список акций в портфеле
    
    Возвращает:
    Матрицу ковариации-дисперсии в виде numpy матрицы
    """
    cm = np.array(c)
    vcv = []
    
    for eachStock in stocksInPortfolio:
        row = []
        for ticker in stocksInPortfolio:
            if eachStock == ticker:
                # Дисперсия: квадрат стандартного отклонения
                # Формула: σ² = (стандартное отклонение)²
                variance = math.pow(stdv[ticker], 2)
                row.append(variance)
            else:
                # Ковариация: ρ_ij * σ_i * σ_j
                # Формула: σ_ij = ρ_ij * σ_i * σ_j
                cov = (stdv[ticker] * stdv[eachStock] * 
                       cm[tickersCorr.index(ticker)][tickersCorr.index(eachStock)])
                row.append(cov)
        vcv.append(row)
    
    vcvmat = np.mat(vcv)
    return vcvmat

# Пример использования
print("Матрица ковариации-дисперсии для портфеля [ABC, XYZ]:")
print(varCovarMatrix(["ABC", "XYZ"]))

### Объяснение формул

В коде выше мы используем следующие математические формулы:

1. **Дисперсия**: $\sigma^2 = (\text{стандартное отклонение})^2$
   
   Для каждой акции на главной диагонали матрицы мы вычисляем дисперсию как квадрат её стандартного отклонения.

2. **Ковариация**: $\sigma_{ij} = \rho_{ij} \cdot \sigma_i \cdot \sigma_j$
   
   Для пар различных акций мы вычисляем ковариацию как произведение коэффициента корреляции и стандартных отклонений обеих акций.

## Реальный пример с российскими акциями

Теперь применим этот подход к реальным данным с Московской биржи.

In [ ]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Функция для получения данных с Московской биржи
def get_moex_data(ticker, days=365):
    """
    Получение исторических данных с Московской биржи
    
    Параметры:
    ticker - тикер акции
    days - количество дней для получения данных
    
    Возвращает:
    DataFrame с историческими ценами
    """
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    url = f"https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/{ticker}.json"
    params = {
        'from': start_date.strftime('%Y-%m-%d'),
        'till': end_date.strftime('%Y-%m-%d'),
        'start': 0
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        # Извлекаем данные
        columns = data['history']['columns']
        rows = data['history']['data']
        
        df = pd.DataFrame(rows, columns=columns)
        df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
        df.set_index('TRADEDATE', inplace=True)
        df.sort_index(inplace=True)
        
        return df[['CLOSE']]
    except Exception as e:
        print(f"Ошибка при получении данных для {ticker}: {e}")
        return None

# Получаем данные для нескольких российских акций
tickers = ['SBER', 'GAZP', 'LKOH', 'MGNT']
prices = {}

for ticker in tickers:
    print(f"Получение данных для {ticker}...")
    data = get_moex_data(ticker)
    if data is not None:
        prices[ticker] = data
        print(f"Получено {len(data)} записей для {ticker}")
    else:
        print(f"Не удалось получить данные для {ticker}")

# Объединяем все данные в один DataFrame
all_prices = pd.concat([prices[ticker] for ticker in tickers if ticker in prices], 
                       axis=1, join='inner')
all_prices.columns = tickers

print("\nПервые 5 строк объединенных данных:")
print(all_prices.head())

In [ ]:
# Рассчитываем дневные доходности
returns = all_prices.pct_change().dropna()

# Визуализируем доходности
plt.figure(figsize=(12, 8))
for ticker in tickers:
    plt.plot(returns.index, returns[ticker], label=ticker, alpha=0.7)
plt.title('Дневные доходности российских акций')
plt.xlabel('Дата')
plt.ylabel('Доходность')
plt.legend()
plt.grid(True)
plt.show()

# Рассчитываем стандартные отклонения
std_devs = returns.std()
print("Стандартные отклонения доходностей:")
print(std_devs)

# Рассчитываем матрицу корреляции
correlation_matrix = returns.corr()
print("\nМатрица корреляции:")
print(correlation_matrix)

In [ ]:
# Создаем функцию для построения матрицы ковариации-дисперсии
def build_variance_covariance_matrix(tickers, std_devs, correlation_matrix):
    """
    Построение матрицы ковариации-дисперсии
    
    Параметры:
    tickers - список тикеров
    std_devs - стандартные отклонения
    correlation_matrix - матрица корреляции
    
    Возвращает:
    Матрицу ковариации-дисперсии
    """
    n = len(tickers)
    vcv_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i == j:
                # Диагональные элементы - дисперсии
                # Формула: σ² = (стандартное отклонение)²
                vcv_matrix[i, j] = std_devs[tickers[i]] ** 2
            else:
                # Недиагональные элементы - ковариации
                # Формула: σ_ij = ρ_ij * σ_i * σ_j
                vcv_matrix[i, j] = (correlation_matrix.iloc[i, j] * 
                                   std_devs[tickers[i]] * 
                                   std_devs[tickers[j]])
    
    return pd.DataFrame(vcv_matrix, index=tickers, columns=tickers)

# Строим матрицу ковариации-дисперсии
vcv_matrix = build_variance_covariance_matrix(tickers, std_devs, correlation_matrix)
print("Матрица ковариации-дисперсии:")
print(vcv_matrix)

# Визуализируем матрицу
plt.figure(figsize=(10, 8))
plt.imshow(vcv_matrix, cmap='viridis', interpolation='nearest')
plt.colorbar(label='Ковариация/Дисперсия')
plt.xticks(range(len(tickers)), tickers)
plt.yticks(range(len(tickers)), tickers)
plt.title('Матрица ковариации-дисперсии российских акций')
plt.show()

## Выводы

На основе анализа российских акций мы можем сделать следующие выводы:

1. **Диверсификация портфеля**: Матрица ковариации-дисперсии показывает, как акции движутся вместе. Низкие или отрицательные значения ковариации указывают на возможности диверсификации.

2. **Риск портфеля**: Диагональные элементы матрицы (дисперсии) представляют индивидуальный риск каждой акции, в то время как недиагональные элементы (ковариации) показывают, как акции взаимодействуют друг с другом.

3. **Управление риском**: Понимание структуры ковариации между активами позволяет более эффективно управлять риском портфеля, минимизируя общую дисперсию портфеля при заданном уровне ожидаемой доходности.

4. **Практическое применение**: Матрица ковариации-дисперсии является фундаментальным инструментом в современной теории портфеля и используется для оптимизации портфеля, расчета Value at Risk (VaR) и других задач управления рисками.

Использование официального API Московской биржи обеспечивает актуальные и надежные данные для анализа, что особенно важно для принятия инвестиционных решений на российском рынке.